# Cloud detection benchmark: classical methods vs. OmniCloudMask

This notebook scores several cloud detectors against our hand-checked masks (`mask_gt.png`).

**What it does**
1. Loads every crop on Drive that has both `image.png` and `mask_gt.png`.
2. Runs four classical methods (rules based on what clouds look like) and the pretrained model OmniCloudMask (OCM).
3. Picks each classical method's threshold on the **val** images, then reports final scores on the **test** images.
4. Shows tables, a smoke check, and pictures of where each method goes wrong.

**Mask values:** `255` = cloud, `0` = not cloud, `64` = smoke, `128` = unsure.
Smoke and unsure pixels are left out of the main scores; smoke gets its own check.

**To run:** Runtime → Run all. You only need to change the settings cell if your Drive folder is somewhere else.

## The methods and where they come from

| Method | Idea | Source |
|---|---|---|
| Brightness + whiteness | Clouds are bright **and** colorless (R, G, B about equal) | Zhu & Woodcock (2012), *Object-based cloud and cloud shadow detection in Landsat imagery*, Remote Sensing of Environment 118 (Fmask). Whiteness test from Gómez-Chova et al. (2007), IEEE TGRS 45(12). |
| Dark-channel prior | Clear ground almost always has a dark pixel nearby; haze and cloud wash that out | He, Sun & Tang (2009), *Single Image Haze Removal Using Dark Channel Prior*, CVPR (also IEEE TPAMI 2011) |
| HOT (haze-optimized transform) | Haze and cloud raise blue more than red, compared with clear ground | Zhang, Guindon & Cihlar (2002), *An image transform to characterize and compensate for spatial variations in thin cloud contamination of Landsat images*, Remote Sensing of Environment 82 |
| Otsu threshold | Picks a brightness cutoff for each image automatically, no tuning | Otsu (1979), *A Threshold Selection Method from Gray-Level Histograms*, IEEE Trans. SMC 9(1) |
| OmniCloudMask (pretrained) | Deep-learning cloud model | Wright et al., OmniCloudMask, https://github.com/DPIRD-DMA/OmniCloudMask |

## 1. Settings

In [ ]:
import os

# Where the crops live. Leave as None to search the usual Drive locations.
CROPS_DIR = os.environ.get("CROPS_DIR") or None

# Where OCM's saved predictions live (<crop>.tif files). None = look next to the crops folder.
OCM_PRED_DIR = os.environ.get("OCM_PRED_DIR") or None

# "saved": score the OCM predictions made when the crops were cut (fast, no GPU).
# "live":  run OmniCloudMask now on each crop (needs a GPU runtime, downloads the model).
OCM_MODE = os.environ.get("OCM_MODE", "saved")
OCM_LIVE_RES_M = 10      # resolution OCM sees in live mode (the crops are 0.3 m; OCM was built for 10-50 m)

# Images are shrunk to this size before scoring, so the notebook runs in a few minutes.
# 2048 = full resolution (slower, needs more RAM).
WORK_SIZE = int(os.environ.get("WORK_SIZE", 512))

# Where tables and figures are saved. None = a "benchmark_results" folder next to the crops.
RESULTS_DIR = os.environ.get("RESULTS_DIR") or None

## 2. Setup (Drive + packages)

In [ ]:
import sys, glob, json, time
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    %pip -q install rasterio
    if OCM_MODE == "live":
        %pip -q install omnicloudmask

def find_crops_dir():
    """Look for ursaspace1C_data/crops in the usual places on Drive."""
    tries = [
        "/content/drive/MyDrive/ursaspace1C_data/crops",
        "/content/drive/MyDrive/cvat_ground_truth/ursaspace1C_data/crops",
        "/content/drive/Shareddrives/ursaspace1C_data/crops",
    ]
    for t in tries:
        if os.path.isdir(t):
            return t
    hits = glob.glob("/content/drive/MyDrive/**/ursaspace1C_data/crops", recursive=True)
    if hits:
        return hits[0]
    raise FileNotFoundError(
        "Can't find ursaspace1C_data/crops. In Google Drive, right-click the shared "
        "'ursaspace1C_data' folder > Organize > Add shortcut > My Drive, then run again. "
        "Or set CROPS_DIR in the settings cell.")

CROPS_DIR = CROPS_DIR or find_crops_dir()
ROOT = os.path.dirname(os.path.normpath(CROPS_DIR))
OCM_PRED_DIR = OCM_PRED_DIR or os.path.join(ROOT, "_ocm_predictions")
RESULTS_DIR = RESULTS_DIR or os.path.join(ROOT, "benchmark_results")
os.makedirs(RESULTS_DIR, exist_ok=True)
print("crops:      ", CROPS_DIR)
print("OCM preds:  ", OCM_PRED_DIR, "(found)" if os.path.isdir(OCM_PRED_DIR) else "(not found)")
print("results go: ", RESULTS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu

CLOUD, SMOKE, UNSURE = 255, 64, 128

## 3. Load the crops and masks

In [ ]:
def load_split(split):
    rows = []
    for d in sorted(glob.glob(os.path.join(CROPS_DIR, split, "*"))):
        img_p, gt_p = os.path.join(d, "image.png"), os.path.join(d, "mask_gt.png")
        if os.path.exists(img_p) and os.path.exists(gt_p):
            rows.append(dict(name=os.path.basename(d), split=split, img=img_p, gt=gt_p))
    return rows

def shrink_rgb(path):
    im = Image.open(path).convert("RGB")
    if im.size != (WORK_SIZE, WORK_SIZE):
        im = im.resize((WORK_SIZE, WORK_SIZE), Image.BILINEAR)
    return np.asarray(im)

def shrink_mask(path):
    m = Image.open(path)
    if m.size != (WORK_SIZE, WORK_SIZE):
        m = m.resize((WORK_SIZE, WORK_SIZE), Image.NEAREST)
    return np.asarray(m)

t0 = time.time()
DATA = {}
for split in ("val", "test"):
    items = load_split(split)
    for it in items:
        it["rgb"] = shrink_rgb(it["img"])
        it["gt"] = shrink_mask(it["gt"])
    DATA[split] = items
    print(f"{split}: {len(items)} crops with masks")
print(f"loaded in {time.time() - t0:.0f}s")
assert DATA["val"] and DATA["test"], "Need masks in both val and test."

def label_summary(items):
    g = np.stack([it["gt"] for it in items])
    return {"cloud %": (g == CLOUD).mean() * 100, "smoke %": (g == SMOKE).mean() * 100,
            "unsure %": (g == UNSURE).mean() * 100, "not cloud %": (g == 0).mean() * 100}
pd.DataFrame({s: label_summary(DATA[s]) for s in DATA}).round(1)

## 4. The classical methods

Each method turns an image into a **score** for every pixel (higher = more cloud-like).
A pixel is called cloud when its score is above a threshold. The threshold is picked on **val**.

In [ ]:
def brightness(rgb):
    return rgb.astype(np.float32).mean(axis=2) / 255.0

def whiteness(rgb):
    """Fmask-style whiteness: how far R, G, B are from their average (0 = perfectly grey/white)."""
    x = rgb.astype(np.float32)
    m = x.mean(axis=2) + 1e-6
    return np.abs(x - m[..., None]).sum(axis=2) / m

def dark_channel(rgb, patch=15):
    """He et al. (2009): darkest value across R, G, B, then the darkest in a patch around each pixel."""
    dc = rgb.min(axis=2)
    return ndi.minimum_filter(dc, size=patch).astype(np.float32) / 255.0

def hot(rgb, slope, intercept):
    """Zhang et al. (2002): distance above the clear-sky line of blue vs. red."""
    r = rgb[..., 0].astype(np.float32); b = rgb[..., 2].astype(np.float32)
    return (b - slope * r - intercept) / np.sqrt(1 + slope ** 2) / 255.0

def otsu_mask(rgb):
    br = brightness(rgb)
    return br > threshold_otsu(br)

HOT needs a "clear-sky line": how blue relates to red on clear ground. Zhang et al. fit it on
clear areas picked by hand. We do the same thing using the pixels marked **not cloud** in the val masks.

In [ ]:
def fit_clear_line(items, n=400_000, seed=0):
    rng = np.random.default_rng(seed)
    R, B = [], []
    for it in items:
        clear = it["gt"] == 0
        r = it["rgb"][..., 0][clear]; b = it["rgb"][..., 2][clear]
        if r.size:
            k = rng.choice(r.size, min(r.size, n // len(items)), replace=False)
            R.append(r[k]); B.append(b[k])
    R = np.concatenate(R).astype(np.float64); B = np.concatenate(B).astype(np.float64)
    slope, intercept = np.polyfit(R, B, 1)
    return slope, intercept

HOT_SLOPE, HOT_INTERCEPT = fit_clear_line(DATA["val"])
print(f"clear-sky line: blue = {HOT_SLOPE:.3f} * red + {HOT_INTERCEPT:.1f}")

## 5. OmniCloudMask predictions

In [ ]:
def read_ocm_saved(name):
    import rasterio
    p = os.path.join(OCM_PRED_DIR, name + ".tif")
    if not os.path.exists(p):
        return None
    with rasterio.open(p) as s:
        a = s.read(1)
    m = Image.fromarray(a)
    if m.size != (WORK_SIZE, WORK_SIZE):
        m = m.resize((WORK_SIZE, WORK_SIZE), Image.NEAREST)
    return np.asarray(m)

_OCM_MODEL_READY = False
def run_ocm_live(img_path):
    """Run OmniCloudMask on one crop, the same way the crops were drafted:
    red + green bands, zeros in place of NIR, shrunk to OCM_LIVE_RES_M."""
    from omnicloudmask import predict_from_array
    full = np.asarray(Image.open(img_path).convert("RGB"))
    px_m = 0.305  # crop pixel size in metres (Maxar visual product)
    side = max(64, int(round(full.shape[0] * px_m / OCM_LIVE_RES_M / 32)) * 32)
    small = np.asarray(Image.fromarray(full).resize((side, side), Image.BOX)).astype(np.float32)
    arr = np.stack([small[..., 0], small[..., 1], np.zeros_like(small[..., 0])])
    ps = min(side, 1000) // 32 * 32
    m = np.asarray(predict_from_array(arr, patch_size=ps, patch_overlap=min(300, ps // 3),
                                      apply_no_data_mask=False))
    m = m.reshape(m.shape[-2:]).astype(np.uint8)
    return np.asarray(Image.fromarray(m).resize((WORK_SIZE, WORK_SIZE), Image.NEAREST))

t0 = time.time()
for split in DATA:
    for it in DATA[split]:
        it["ocm"] = run_ocm_live(it["img"]) if OCM_MODE == "live" else read_ocm_saved(it["name"])
have = sum(it["ocm"] is not None for s in DATA for it in DATA[s])
total = sum(len(DATA[s]) for s in DATA)
print(f"OCM ({OCM_MODE}): {have}/{total} crops in {time.time() - t0:.0f}s")
OCM_OK = have == total
if not OCM_OK:
    print("OCM will be skipped. Copy _ocm_predictions next to the crops folder, or set OCM_MODE = 'live'.")

# OCM classes: 0 clear, 1 thick cloud, 2 thin cloud, 3 shadow, 4 smoke, 5 unsure
def ocm_cloud(it):
    return np.isin(it["ocm"], (1, 2))

## 6. Scoring helpers

Scores are counted over all pixels of a split together (not averaged per image), on cloud vs. not-cloud
pixels only. **Precision** = of the pixels called cloud, how many really are. **Recall** = of the real
cloud, how much was found. **F1** balances the two. **IoU** = overlap ÷ union of predicted and real cloud.

In [ ]:
def counts(pred, gt):
    valid = (gt == CLOUD) | (gt == 0)
    p, t = pred & valid, (gt == CLOUD)
    tp = np.sum(p & t); fp = np.sum(p & ~t); fn = np.sum(~p & t & valid)
    return tp, fp, fn

def metrics(tp, fp, fn):
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    iou = tp / (tp + fp + fn) if tp + fp + fn else 0.0
    return dict(IoU=iou, F1=f1, precision=prec, recall=rec)

def smoke_as_cloud(pred, gt):
    s = gt == SMOKE
    return np.sum(pred & s), np.sum(s)

## 7. Pick thresholds on val

For each method we try many thresholds on the val images and keep the one with the best F1.
This is done with histograms, so trying hundreds of thresholds is quick.

In [ ]:
BINS = np.linspace(-0.5, 1.0, 301)   # score range covered by all methods

def score_hist(score_fn, items):
    """Histogram of scores for real-cloud and real-clear pixels."""
    hc = np.zeros(len(BINS) - 1); hn = np.zeros(len(BINS) - 1)
    for it in items:
        s = score_fn(it["rgb"]); g = it["gt"]
        hc += np.histogram(s[g == CLOUD], BINS)[0]
        hn += np.histogram(s[g == 0], BINS)[0]
    return hc, hn

def best_threshold(hc, hn):
    tp = hc[::-1].cumsum()[::-1]; fp = hn[::-1].cumsum()[::-1]; fn = hc.sum() - tp
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)
    i = int(np.argmax(f1))
    return BINS[i], f1[i]

SCORES = {
    "Dark-channel prior": lambda rgb: dark_channel(rgb),
    "HOT": lambda rgb: hot(rgb, HOT_SLOPE, HOT_INTERCEPT),
}
THRESH = {}
for name, fn in SCORES.items():
    THRESH[name], f1 = best_threshold(*score_hist(fn, DATA["val"]))
    print(f"{name:22s} threshold {THRESH[name]:.3f}  (val F1 {f1:.3f})")

# Brightness + whiteness has two thresholds, so search a 2-D grid.
B_BINS = np.linspace(0, 1, 101); W_BINS = np.linspace(0, 1.5, 76)
H2c = np.zeros((100, 75)); H2n = np.zeros((100, 75))
for it in DATA["val"]:
    br, wh, g = brightness(it["rgb"]), np.minimum(whiteness(it["rgb"]), 1.499), it["gt"]
    H2c += np.histogram2d(br[g == CLOUD], wh[g == CLOUD], [B_BINS, W_BINS])[0]
    H2n += np.histogram2d(br[g == 0], wh[g == 0], [B_BINS, W_BINS])[0]
# cloud = brightness >= b AND whiteness <= w  ->  sum over bright rows, low-whiteness columns
TPg = H2c[::-1].cumsum(0)[::-1].cumsum(1); FPg = H2n[::-1].cumsum(0)[::-1].cumsum(1)
FNg = H2c.sum() - TPg
F1g = 2 * TPg / np.maximum(2 * TPg + FPg + FNg, 1)
bi, wi = np.unravel_index(np.argmax(F1g), F1g.shape)
BW_B, BW_W = B_BINS[bi], W_BINS[wi + 1]
print(f"{'Brightness + whiteness':22s} brightness >= {BW_B:.2f} and whiteness <= {BW_W:.2f}  (val F1 {F1g[bi, wi]:.3f})")

## 8. Run every method

In [ ]:
def predict(method, it):
    rgb = it["rgb"]
    if method == "Brightness + whiteness":
        return (brightness(rgb) >= BW_B) & (whiteness(rgb) <= BW_W)
    if method in SCORES:
        return SCORES[method](rgb) >= THRESH[method]
    if method == "Otsu (no tuning)":
        return otsu_mask(rgb)
    if method == "OmniCloudMask":
        return ocm_cloud(it)
    raise ValueError(method)

METHODS = ["Brightness + whiteness", "Dark-channel prior", "HOT", "Otsu (no tuning)"]
if OCM_OK:
    METHODS.append("OmniCloudMask")

per_image, summary = [], []
for split in ("val", "test"):
    for m in METHODS:
        TP = FP = FN = SC = SN = 0
        for it in DATA[split]:
            p = predict(m, it)
            tp, fp, fn = counts(p, it["gt"]); sc, sn = smoke_as_cloud(p, it["gt"])
            TP += tp; FP += fp; FN += fn; SC += sc; SN += sn
            row = dict(split=split, method=m, crop=it["name"], **metrics(tp, fp, fn),
                       cloud_true_pct=(it["gt"] == CLOUD).mean() * 100, cloud_pred_pct=p.mean() * 100)
            per_image.append(row)
        summary.append(dict(split=split, method=m, **metrics(TP, FP, FN),
                            smoke_called_cloud=(SC / SN if SN else np.nan)))
per_image = pd.DataFrame(per_image); summary = pd.DataFrame(summary)
per_image.to_csv(os.path.join(RESULTS_DIR, "per_image_scores.csv"), index=False)
summary.to_csv(os.path.join(RESULTS_DIR, "summary_scores.csv"), index=False)

## 9. Results

**Test** is the fair score: the thresholds were picked on val and never saw these images.
`smoke_called_cloud` = share of smoke pixels the method wrongly called cloud (lower is better).

In [ ]:
fmt = {c: "{:.3f}" for c in ["IoU", "F1", "precision", "recall", "smoke_called_cloud"]}
print("TEST (final scores)")
display(summary[summary.split == "test"].drop(columns="split").set_index("method").style.format(fmt))
print("VAL (used to pick thresholds)")
display(summary[summary.split == "val"].drop(columns="split").set_index("method").style.format(fmt))

In [ ]:
t = summary[summary.split == "test"].set_index("method")
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(t)); w = 0.2
for i, col in enumerate(["IoU", "F1", "precision", "recall"]):
    ax.bar(x + (i - 1.5) * w, t[col], w, label=col)
ax.set_xticks(x); ax.set_xticklabels(t.index, rotation=15); ax.set_ylim(0, 1)
ax.set_title("Test scores by method"); ax.legend(ncol=4, loc="upper left"); ax.grid(axis="y", alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, "test_scores.png"), dpi=120); plt.show()

## 10. How much the hand-checked masks differ from OCM's drafts

The masks started as OCM drafts and were then corrected by hand. This shows how much was changed:
the share of pixels where the final cloud / not-cloud answer differs from OCM's.

In [ ]:
if OCM_OK:
    rows = []
    for split in DATA:
        for it in DATA[split]:
            ocm_c = ocm_cloud(it); gt_c = it["gt"] == CLOUD
            rows.append(dict(split=split, crop=it["name"], pct_pixels_changed=(ocm_c != gt_c).mean() * 100))
    changed = pd.DataFrame(rows)
    changed.to_csv(os.path.join(RESULTS_DIR, "gt_vs_ocm_changes.csv"), index=False)
    print(f"Average share of pixels changed from OCM's draft: {changed.pct_pixels_changed.mean():.1f}%")
    print(f"Crops with more than 10% of pixels changed: {(changed.pct_pixels_changed > 10).sum()} of {len(changed)}")
    display(changed.groupby("split").pct_pixels_changed.describe().round(1))

## 11. Pictures: where each method goes right and wrong

White = cloud, grey = smoke, black = not cloud. In the method columns:
**green** = correct cloud, **red** = false alarm, **blue** = missed cloud.

In [ ]:
def error_map(pred, gt):
    out = np.zeros(gt.shape + (3,), np.uint8)
    out[(gt == SMOKE) | (gt == UNSURE)] = (90, 90, 90)
    out[pred & (gt == CLOUD)] = (40, 200, 70)
    out[pred & (gt == 0)] = (230, 50, 50)
    out[~pred & (gt == CLOUD)] = (40, 110, 255)
    return out

def gt_view(gt):
    v = np.zeros(gt.shape, np.uint8); v[gt == CLOUD] = 255; v[gt == SMOKE] = 110; v[gt == UNSURE] = 170
    return v

def show_examples(items, n=10, fname="test_examples.png"):
    items = sorted(items, key=lambda it: -(it["gt"] == CLOUD).mean())
    pick = items[:: max(1, len(items) // n)][:n]
    cols = 2 + len(METHODS)
    fig, axes = plt.subplots(len(pick), cols, figsize=(2.1 * cols, 2.1 * len(pick)))
    for r, it in enumerate(pick):
        axes[r, 0].imshow(it["rgb"]); axes[r, 1].imshow(gt_view(it["gt"]), cmap="gray", vmin=0, vmax=255)
        for c, m in enumerate(METHODS):
            axes[r, 2 + c].imshow(error_map(predict(m, it), it["gt"]))
        axes[r, 0].set_ylabel(it["name"], fontsize=8)
    for c, t in enumerate(["image", "hand-checked"] + METHODS):
        axes[0, c].set_title(t, fontsize=8)
    for a in axes.ravel():
        a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.savefig(os.path.join(RESULTS_DIR, fname), dpi=110); plt.show()

show_examples(DATA["test"])

## 12. Hardest test crops for each method

In [ ]:
hard = (per_image[(per_image.split == "test") & (per_image.cloud_true_pct > 1)]
        .sort_values("IoU").groupby("method").head(3)[["method", "crop", "IoU", "cloud_true_pct", "cloud_pred_pct"]])
hard.round(2)

## 13. Settings used (for the write-up)

In [ ]:
settings = dict(WORK_SIZE=WORK_SIZE, OCM_MODE=OCM_MODE, n_val=len(DATA["val"]), n_test=len(DATA["test"]),
                brightness_min=float(BW_B), whiteness_max=float(BW_W),
                dark_channel_threshold=float(THRESH["Dark-channel prior"]), dark_channel_patch=15,
                hot_threshold=float(THRESH["HOT"]), hot_slope=float(HOT_SLOPE), hot_intercept=float(HOT_INTERCEPT))
with open(os.path.join(RESULTS_DIR, "settings.json"), "w") as f:
    json.dump(settings, f, indent=2)
print(json.dumps(settings, indent=2))
print("Saved tables and figures to", RESULTS_DIR)